# Multi Attack Runner
Coordinate PWSCUP2025 attack scripts with reusable helpers and grouping.


In [ ]:
import os
import sys
import subprocess
from pathlib import Path
from shutil import which
from typing import Dict, Optional, Sequence


In [ ]:

TEAM_IDS = tuple(i for i in range(1, 23) if i != 21)


def get_teams():
    '''Return the default sequence of team IDs.'''
    return TEAM_IDS


In [ ]:

    def loop_for_all_teams(
        command_template,
        *,
        teams=None,
        dry_run=False,
        strict=True,
        continue_on_error=False,
        cwd=None
    ):
        '''
        command_template: ['python', 'attack/attack_Ci.py', '...{id:02d}...', ...]
        teams: explicit iterable of team ids. Defaults to get_teams().
        dry_run: True -> only print expanded commands.
        strict: True -> require at least one {id:02d} placeholder.
        continue_on_error: True -> keep going after failures.
        cwd: working directory for subprocess.run.
        '''
        id_indices = [
            i for i, arg in enumerate(command_template)
            if isinstance(arg, str) and "{id:02d}" in arg
        ]
        if strict and not id_indices:
            raise ValueError(f"No {{id:02d}} placeholder found in: {command_template}")

        cmd0 = command_template[:]
        if cmd0 and cmd0[0] in ("python", "python3"):
            cmd0[0] = sys.executable

        exe = cmd0[0]
        if os.path.sep not in exe and which(exe) is None:
            raise RuntimeError(f"Executable not found on PATH: {exe}")

        team_list = get_teams() if teams is None else teams

        for team in team_list:
            cmd = cmd0[:]
            for ind in id_indices:
                cmd[ind] = cmd[ind].format(id=team)

            def is_out_flag(i: int) -> bool:
                if i == 0:
                    return False
                prev = cmd[i - 1]
                return isinstance(prev, str) and (
                    prev in ("-o", "--out", "--out-map", "--out-pred", "--out-conf")
                    or prev.startswith("--out")
                )

            missing_inputs = []
            for i, arg in enumerate(cmd):
                if isinstance(arg, str) and arg.lower().endswith((".csv", ".json")) and not is_out_flag(i):
                    candidate = arg if cwd is None else os.path.join(cwd, arg)
                    if not os.path.exists(candidate):
                        missing_inputs.append(arg)

            print(">>", " ".join(cmd))
            if missing_inputs:
                message = f"[team {team}] Missing input files: {missing_inputs}"
                if continue_on_error:
                    print("!!", message)
                    continue
                raise FileNotFoundError(message)

            if dry_run:
                continue

            try:
                completed = subprocess.run(
                    cmd,
                    check=True,
                    cwd=cwd,
                    capture_output=True,
                    text=True,
                )
                if completed.stdout:
                    print(completed.stdout.strip())
            except subprocess.CalledProcessError as exc:
                print(f"
[ERROR] team {team} command failed with code {exc.returncode}")
                if exc.stdout:
                    print("--- stdout ---")
                    print(exc.stdout.strip())
                if exc.stderr:
                    print("--- stderr ---")
                    print(exc.stderr.strip())
                if not continue_on_error:
                    raise


In [ ]:

    COMMAND_REGISTRY: Dict[str, Dict[str, object]] = {}
    PIPELINE_ORDER: list[str] = []


    def register_command(
        key: str,
        *,
        label: str,
        template: Sequence[str],
        category: str,
        description: str = "",
        cwd: Optional[str] = None,
        continue_on_error: bool = False,
    ) -> None:
        COMMAND_REGISTRY[key] = {
            "label": label,
            "template": list(template),
            "category": category,
            "description": description,
            "cwd": cwd,
            "continue_on_error": continue_on_error,
        }
        if key not in PIPELINE_ORDER:
            PIPELINE_ORDER.append(key)


    def run_command(
        key: str,
        *,
        label: str,
        template: Sequence[str],
        category: str,
        description: str = "",
        teams: Optional[Sequence[int]] = None,
        dry_run: bool = False,
        continue_on_error: bool = False,
        cwd: Optional[str] = None,
    ) -> None:
        register_command(
            key,
            label=label,
            template=template,
            category=category,
            description=description,
            cwd=cwd,
            continue_on_error=continue_on_error,
        )
        print(f"
=== {label} ({key}) ===")
        loop_for_all_teams(
            template,
            teams=teams,
            dry_run=dry_run,
            continue_on_error=continue_on_error,
            cwd=cwd,
        )


    def rerun_command(
        key: str,
        *,
        teams: Optional[Sequence[int]] = None,
        dry_run: bool = False,
        continue_on_error: Optional[bool] = None,
        cwd: Optional[str] = None,
    ) -> None:
        if key not in COMMAND_REGISTRY:
            raise KeyError(f"Unknown command key: {key}")
        spec = COMMAND_REGISTRY[key]
        effective_continue = (
            spec.get("continue_on_error", False)
            if continue_on_error is None
            else continue_on_error
        )
        print(f"
=== {spec['label']} ({key}) ===")
        loop_for_all_teams(
            spec["template"],
            teams=teams,
            dry_run=dry_run,
            continue_on_error=effective_continue,
            cwd=cwd or spec.get("cwd"),
        )


    def list_commands(category: Optional[str] = None) -> None:
        for key in PIPELINE_ORDER:
            spec = COMMAND_REGISTRY.get(key)
            if spec is None:
                continue
            if category and spec["category"] != category:
                continue
            line = f"{key:>24}  {spec['label']} [{spec['category']}]"
            print(line)
            desc = spec.get("description")
            if desc:
                print(f"    {desc}")


    def run_pipeline(
        keys: Optional[Sequence[str]] = None,
        *,
        teams: Optional[Sequence[int]] = None,
        dry_run: bool = False,
        continue_on_error: Optional[bool] = None,
    ) -> None:
        sequence = keys if keys is not None else PIPELINE_ORDER
        for key in sequence:
            rerun_command(
                key,
                teams=teams,
                dry_run=dry_run,
                continue_on_error=continue_on_error,
            )


    TEAM_22_ONLY = (22,)


In [ ]:

def find_project_root(start: Path) -> Path:
    markers = ("requirements.txt", "GUIDE_FOR_BEGINNERS.md")
    for candidate in [start, *start.parents]:
        if all((candidate / marker).exists() for marker in markers):
            return candidate
    return start


PROJECT_ROOT = find_project_root(Path.cwd())
os.chdir(PROJECT_ROOT)
print(f"Working directory set to: {PROJECT_ROOT}")


## Mode Configuration
Adjust dataset prefixes for prep or contest runs using the constants below.


In [ ]:

MODE_PRESETS = {
    "prep": {
        "original": "B",
        "anon": "C",
        "model": "D",
        "variants": ["3"],
    },
    "contest": {
        "original": "BB",
        "anon": "CC",
        "model": "DD",
        "variants": ["1", "2", "3"],
    },
}

mode = "prep"  # change to "contest" for contest datasets
MODE_CONFIG = MODE_PRESETS[mode]
mode_original = MODE_CONFIG["original"]
mode_anon = MODE_CONFIG["anon"]
mode_model = MODE_CONFIG["model"]
mode_variants = MODE_CONFIG["variants"]


## Preprocessing


In [ ]:

run_command(
    key="fix_csv",
    label="Fix anonymized CSV files",
    category="preprocess",
    description="Check column ranges and emit *_fix CSV files.",
    template=[
        "python",
        "util/check_and_fix_csv.py",
        f"in/PWSCUP2025_Pre_Data_for_Attack/{mode_anon}{{id:02d}}.csv",
        "data/pre_columns_range.json",
        f"in/PWSCUP2025_Pre_Data_for_Attack/{mode_anon}{{id:02d}}_fix.csv",
    ],
)


## Ci Attacks


In [ ]:

run_command(
    key="ci_original",
    label="Ci attack (original)",
    category="ci",
    description="Baseline Ci attack using original samples.",
    template=[
        "python",
        "attack/attack_Ci.py",
        f"in/PWSCUP2025_Pre_Data_for_Attack/{mode_original}{{id:02d}}.csv",
        f"in/PWSCUP2025_Pre_Data_for_Attack/{mode_anon}{{id:02d}}_fix.csv",
        "-o",
        f"out_attack/{mode_anon}{{id:02d}}_inferred.csv",
    ],
)


In [ ]:

run_command(
    key="ci_extended",
    label="Ci attack (extended)",
    category="ci",
    description="Extended Ci attack with single-neighbour output.",
    template=[
        "python",
        "attack/attack_Ci_ex.py",
        f"in/PWSCUP2025_Pre_Data_for_Attack/{mode_original}{{id:02d}}.csv",
        f"in/PWSCUP2025_Pre_Data_for_Attack/{mode_anon}{{id:02d}}_fix.csv",
        "-o",
        f"out_attack/{mode_anon}{{id:02d}}_inferred_ex.csv",
        "-k",
        "1",
    ],
)


In [ ]:

run_command(
    key="ci_knn",
    label="Ci attack (k-NN)",
    category="ci",
    description="k-nearest Ci attack (mode=nn, k=5).",
    template=[
        "python",
        "attack/attack_Ci_ex_greedy.py",
        f"in/PWSCUP2025_Pre_Data_for_Attack/{mode_anon}{{id:02d}}_fix.csv",
        f"in/PWSCUP2025_Pre_Data_for_Attack/{mode_original}{{id:02d}}.csv",
        "-m",
        "nn",
        "-k",
        "5",
        "-o",
        f"out_attack/{mode_anon}{{id:02d}}_inferred_ex_greedy_k5_nn.csv",
    ],
)


In [ ]:

run_command(
    key="ci_greedy",
    label="Ci attack (greedy)",
    category="ci",
    description="Greedy Ci attack with k=300 for exhaustive matching.",
    template=[
        "python",
        "attack/attack_Ci_ex_greedy.py",
        f"in/PWSCUP2025_Pre_Data_for_Attack/{mode_anon}{{id:02d}}_fix.csv",
        f"in/PWSCUP2025_Pre_Data_for_Attack/{mode_original}{{id:02d}}.csv",
        "-m",
        "greedy",
        "-k",
        "300",
        "-o",
        f"out_attack/{mode_anon}{{id:02d}}_inferred_ex_greedy_k300_greedy.csv",
        "--out-map",
        f"out_attack/{mode_anon}{{id:02d}}_matchmap_k300.csv",
    ],
)


## Di Attacks


In [ ]:

run_command(
    key="di_original",
    label="Di attack (original)",
    category="di",
    description="Baseline Di attack using provided model predictions.",
    template=[
        "python",
        "attack/attack_Di.py",
        f"in/PWSCUP2025_Pre_Data_for_Attack/{mode_model}{{id:02d}}.json",
        f"in/PWSCUP2025_Pre_Data_for_Attack/{mode_original}{{id:02d}}.csv",
    ],
)


In [ ]:

run_command(
    key="di_extended",
    label="Di attack (extended)",
    category="di",
    description="Extended Di attack with prediction and confidence outputs.",
    template=[
        "python",
        "attack/attack_Di_ex.py",
        f"in/PWSCUP2025_Pre_Data_for_Attack/{mode_model}{{id:02d}}.json",
        f"in/PWSCUP2025_Pre_Data_for_Attack/{mode_original}{{id:02d}}.csv",
        "--out-pred",
        "out_attack/inferred_membership1_{id:02d}_ex.csv",
        "--out-conf",
        "out_attack/inferred_membership2_{id:02d}_ex.csv",
    ],
)


## Combination Attacks


In [ ]:

run_command(
    key="combi_original",
    label="Combination attack (original)",
    category="combination",
    description="Combine Ci and Di original outputs.",
    template=[
        "python",
        "attack/attack_example_ex.py",
        "--Ai_csv",
        f"in/PWSCUP2025_Pre_Data_for_Attack/{mode_original}{{id:02d}}.csv",
        "-o",
        f"out_attack/Fij_{{id:02d}}.csv",
        f"out_attack/{mode_anon}{{id:02d}}_inferred.csv",
        "out_attack/inferred_membership1_{id:02d}_ex.csv",
        "out_attack/inferred_membership2_{id:02d}_ex.csv",
    ],
)


In [ ]:

run_command(
    key="combi_extended",
    label="Combination attack (extended)",
    category="combination",
    description="Extended combination attack with limit 10000.",
    template=[
        "python",
        "attack/attack_example_ex.py",
        "--Ai_csv",
        f"in/PWSCUP2025_Pre_Data_for_Attack/{mode_original}{{id:02d}}.csv",
        "-o",
        "in/Fij_{id:02d}.csv",
        "-l",
        "10000",
        f"out_attack/{mode_anon}{{id:02d}}_inferred_ex_greedy_k300_greedy.csv",
        "out_attack/inferred_membership1_{id:02d}_ex.csv",
        "out_attack/inferred_membership2_{id:02d}_ex.csv",
    ],
)


## Scoring Strategies


In [ ]:

run_command(
    key="new_dici_scoring",
    label="New Di->Ci scoring",
    category="scoring",
    description="Rank Di candidates by Ci distance and prediction error (union mode).",
    template=[
        "python",
        "attack/new_attackDi_Ci.py",
        f"in/PWSCUP2025_Pre_Data_for_Attack/{mode_original}{{id:02d}}.csv",
        f"in/PWSCUP2025_Pre_Data_for_Attack/{mode_anon}{{id:02d}}_fix.csv",
        f"in/PWSCUP2025_Pre_Data_for_Attack/{mode_model}{{id:02d}}.json",
        "--pred-threshold",
        "0.5",
        "--conf-threshold",
        "0.25",
        "--mode",
        "intersection",
        "-k",
        "5",
        "--w-conf",
        "1.0",
        "--topn",
        "10000",
        "-o",
        "out_attack/Fij_new_{id:02d}.csv",
        "--out-rank",
        "out_attack/Fij_new_{id:02d}_rank.csv",
    ],
)


In [ ]:

run_command(
    key="new_dici_scoring_greedy",
    label="New Di->Ci scoring (greedy)",
    category="scoring",
    description="Greedy expansion variant of the new Di->Ci scoring attack.",
    template=[
        "python",
        "attack/new_attackDi_Ci_greedy.py",
        f"in/PWSCUP2025_Pre_Data_for_Attack/{mode_original}{{id:02d}}.csv",
        f"in/PWSCUP2025_Pre_Data_for_Attack/{mode_anon}{{id:02d}}_fix.csv",
        f"in/PWSCUP2025_Pre_Data_for_Attack/{mode_model}{{id:02d}}.json",
        "--pred-threshold",
        "0.5",
        "--conf-threshold",
        "0.25",
        "--mode",
        "intersection",
        "--w-conf",
        "1.0",
        "--topn",
        "10000",
        "-o",
        "out_attack/Fij_new_greedy_{id:02d}.csv",
        "--out-rank",
        "out_attack/Fij_new_greedy_{id:02d}_rank.csv",
        "--out-map",
        f"out_attack/{mode_anon}{{id:02d}}_matchmap_greedy.csv",
    ],
)


In [ ]:

run_command(
    key="ci_di_independent",
    label="Independent Ci + Di scoring",
    category="scoring",
    description="Combine Ci distance and Di prediction error independently.",
    template=[
        "python",
        "attack/attack_Ci_Di_independent.py",
        f"in/PWSCUP2025_Pre_Data_for_Attack/{mode_original}{{id:02d}}.csv",
        f"in/PWSCUP2025_Pre_Data_for_Attack/{mode_anon}{{id:02d}}_fix.csv",
        f"in/PWSCUP2025_Pre_Data_for_Attack/{mode_model}{{id:02d}}.json",
        "--w-conf",
        "1.0",
        "--k-hint",
        "300",
        "--topn",
        "10000",
        "-o",
        "out_attack/Fij_independent_{id:02d}.csv",
        "--out-rank",
        "out_attack/Fij_independent_{id:02d}_rank.csv",
        "--out-map",
        f"out_attack/{mode_anon}{{id:02d}}_matchmap_independent.csv",
    ],
)


In [ ]:

run_command(
    key="ci_hungarian",
    label="Ci attack (Hungarian)",
    category="ci",
    description="Hungarian assignment variant of the Ci attack (auto mode, k=300).",
    template=[
        "python",
        "attack/attack_Ci_hungarian.py",
        f"in/PWSCUP2025_Pre_Data_for_Attack/{mode_original}{{id:02d}}.csv",
        f"in/PWSCUP2025_Pre_Data_for_Attack/{mode_anon}{{id:02d}}_fix.csv",
        "-m",
        "auto",
        "-k",
        "300",
        "-o",
        f"out_attack/{mode_anon}{{id:02d}}_inferred_hungarian_auto_k300.csv",
        "--out-map",
        f"out_attack/{mode_anon}{{id:02d}}_matchmap_hungarian_auto_k300.csv",
    ],
)


In [ ]:

run_command(
    key="new_dici_hungarian",
    label="New Di->Ci scoring (Hungarian)",
    category="scoring",
    description="Apply Di->Ci scoring after Hungarian Ci matching.",
    template=[
        "python",
        "attack/attackDi_Ci_hungarian.py",
        f"in/PWSCUP2025_Pre_Data_for_Attack/{mode_original}{{id:02d}}.csv",
        f"in/PWSCUP2025_Pre_Data_for_Attack/{mode_anon}{{id:02d}}_fix.csv",
        f"in/PWSCUP2025_Pre_Data_for_Attack/{mode_model}{{id:02d}}.json",
        "--pred-threshold",
        "0.5",
        "--conf-threshold",
        "0.25",
        "--mode",
        "intersection",
        "--hung-mode",
        "auto",
        "-k",
        "300",
        "--w-conf",
        "1.0",
        "--topn",
        "10000",
        "-o",
        "out_attack/Fij_new_hung_{id:02d}.csv",
        "--out-rank",
        "out_attack/Fij_new_hung_{id:02d}_rank.csv",
        "--out-map",
        f"out_attack/{mode_anon}{{id:02d}}_matchmap_hungarian_used.csv",
    ],
)


In [ ]:

run_command(
    key="allci_alldi_hungarian",
    label="AllCi + AllDi (Hungarian)",
    category="scoring",
    description="Aggregate all Ci and Di candidates with Hungarian scoring.",
    template=[
        "python",
        "attack/attack_allCi_allDi_hungarian.py",
        f"in/PWSCUP2025_Pre_Data_for_Attack/{mode_original}{{id:02d}}.csv",
        f"in/PWSCUP2025_Pre_Data_for_Attack/{mode_anon}{{id:02d}}_fix.csv",
        f"in/PWSCUP2025_Pre_Data_for_Attack/{mode_model}{{id:02d}}.json",
        "--hung-mode",
        "auto",
        "-k",
        "300",
        "--w-dist",
        "1.0",
        "--w-conf",
        "1.0",
        "--topn",
        "10000",
        "-o",
        "out_attack/Fij_all_hungarian_{id:02d}.csv",
        "--out-rank",
        "out_attack/Fij_all_hungarian_{id:02d}_rank.csv",
        "--out-map",
        f"out_attack/{mode_anon}{{id:02d}}_matchmap_all_hungarian.csv",
    ],
)


## Pipeline Controls
- Call list_commands() to view registered steps.
- Use the toggles below to preview or rerun commands.


In [ ]:
RUN_PIPELINE = False  # set to True to execute the recorded sequence
DRY_RUN = True        # set to False to launch subprocesses
TARGET_TEAMS = None   # e.g., TEAM_22_ONLY for quick checks

if RUN_PIPELINE:
    run_pipeline(teams=TARGET_TEAMS, dry_run=DRY_RUN)


## Dataset Checklist
- Ensure the in/ directory exists at the project root.
- Place PWSCUP2025_Pre_Data_for_Attack archives under in/ and extract them to in/PWSCUP2025_Pre_Data_for_Attack/.
- Example layout: in/PWSCUP2025_Pre_Data_for_Attack/A01.csv


## Answer Generation
Example: python evaluation/gen_ans.py in/PWSCUP2025_Pre_Data_for_Attack/A22.csv in/B22_3.csv -o in/Z22.csv


## Evaluation
Example: python evaluation/check_ans.py out_attack/Fij_all_hungarian_22.csv in/Z22.csv
